## Demo of client side Union of hotset dataset as view over Kafka through ISK and coldset dataset on MiniIO

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

spark



### ISK view over Kafka has multiple topics including two topics with transactions data:
    
 - **transactions_old** with static set of data of 500000 events sorted by TransactionTime - that can be loaded into the cold set on MiniIO for demo purposes
 - **transactions** with dynamic set of data - starting with 50 events and gets new event every second.



In [ ]:
%%sql

USE isk.isk

In [ ]:
%%sql

show tables;

In [ ]:
%%sql

DESCRIBE transactions;

In [ ]:
%%sql
--  Load transactions_old events into the coldset using CTAS operation, partitioned by TransactionTime hour.
CREATE TABLE minio.data.transactions
          USING iceberg
          PARTITIONED BY (HOUR(TransactionTime))
          TBLPROPERTIES('format-version'='2')
 AS SELECT * FROM isk.isk.transactions_old;

In [ ]:
%%sql
-- Verify number of rows written to the cold set
select count (*) from  minio.data.transactions;

In [ ]:
%%sql
-- Verify number of rows on the hot set in transactions topic 
-- Note that we have to use an where clause to force scan over data on topic - otherwise approximation in metadata is used.
select * from isk.isk.transactions.snapshots;

In [ ]:
%%sql
-- Verify number of rows on the hot set in transactions topic 
-- Note that we have to use an where clause to force scan over data on topic - otherwise approximation in metadata is used.
select count (*) from  isk.isk.transactions where transactionTime < Now();

In [ ]:
%%sql
-- Total deposits - withdrawals per branch - without time bounds across cold and hotset   
    
select round(sum( 
case 
    when t.transactiontype='Withdrawal' THEN t.transactionamount*(-1)
    else t.transactionamount
END
),2) as total_per_branch,
b.branchname  
from 
(SELECT * FROM minio.data.transactions m
UNION
select * from isk.isk.transactions i) t    
    
    join branches b on t.branchid=b.branchid 
--    where t.transactiontime between '2025-04-21 00:00:00' AND '2025-04-21 23:59:59'
    group by b.branchname order by b.branchname asc;

In [ ]:
%%sql
-- Count of transaction events across cold and hot set 

select count (*) from (SELECT * FROM minio.data.transactions m
UNION
select * from isk.isk.transactions i WHERE i.transactionTime> (SELECT max(transactionTime) FROM minio.data.transactions));

In [ ]:
%%sql
-- Latest 100 events on the hotset

select * from isk.isk.transactions ORDER BY transactionTime DESC LIMIT 100;